## **QMML Christmas Hackathon**

This hackathon uses a Christmas sales dataset with 8,000 training transactions and 2,000 test transactions. Your goal is to predict whether purchased items will be returned based on features like product category, customer demographics, weather conditions, promotions, gift wrapping, and more.

This baseline notebook achieves approximately 50% accuracy (barely better than random guessing). Your goal is to improve on this!

**Prizes:**
- Prize 1: Best accuracy on test set
- Prize 2: Best visualizations and charts

**Submission Requirements:**

You must submit a CSV file with exactly two columns: `TransactionID` and `ReturnFlag` (0 or 1), containing predictions for all 2,000 test transactions (IDs 8001-10000). Your submission will be evaluated on accuracy against the hidden test labels. This starter notebook already produced such csv file. Please change the file name to represent YOUR name/team name

### **About the Dataset**

**train.csv** (8,000 rows) contains the following features:
- **Transaction Info**: TransactionID, Date, Time
- **Customer Info**: CustomerID, Age, Gender, Location
- **Product Info**: ProductID, ProductName, Category, Quantity, UnitPrice, TotalPrice
- **Purchase Details**: StoreID, OnlineOrderFlag, PaymentType, PromotionApplied, DiscountAmount, GiftWrap
- **Shipping Info**: ShippingMethod, DeliveryTime (for online orders)
- **Context**: Weather, Event (e.g., Black Friday, Christmas Market)
- **Feedback**: CustomerSatisfaction (1-5 scale)
- ***Target***: ReturnFlag (True/False - what you're predicting!)

**test.csv** (2,000 rows) contains all the same features except ReturnFlag, which you need to predict.

**Key Notes:**
- Some features have missing values (e.g., StoreID for online orders, ShippingMethod for in-store purchases)
- Dates range from 2018-2023 during the November-December holiday shopping season
- Categories include: Electronics, Clothing, Toys, Food, Decorations

### **Setup and Imports**

In [11]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression

# Try other models:
# from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
# from xgboost import XGBClassifier
# from sklearn.svm import SVC
# from sklearn.neighbors import KNeighborsClassifier

# For visualization (Prize 2):
# import matplotlib.pyplot as plt
# import seaborn as sns

### **Load Data**

In [12]:
TRAIN_PATH = "./data/train.csv"
TEST_PATH = "./data/test.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df.head()

Train shape: (8000, 25)
Test shape: (2000, 24)


,TransactionID,Date,Time,CustomerID,Age,Gender,Location,StoreID,OnlineOrderFlag,ProductID,...,PaymentType,PromotionApplied,DiscountAmount,GiftWrap,ShippingMethod,DeliveryTime,Weather,Event,CustomerSatisfaction,ReturnFlag
0,1,12/24/2020,7:27:59,441,27,Other,City_15,NaN,True,106,...,Credit Card,False,0.0,False,Standard,5.0,Snowy,NaN,5,False
1,2,11/18/2022,14:36:39,340,43,Male,City_13,NaN,True,816,...,Credit Card,True,0.0,True,Express,3.0,Sunny,NaN,2,True
2,3,12/26/2019,20:23:50,31,25,Other,City_7,92.0,False,508,...,Credit Card,False,0.0,False,NaN,NaN,Rainy,Christmas Market,4,False
3,4,11/13/2018,23:08:08,39,64,Male,City_20,100.0,False,710,...,Debit Card,False,0.0,True,NaN,NaN,Rainy,NaN,1,True
4,5,12/13/2020,4:38:08,344,26,Other,City_10,90.0,False,687,...,Cash,False,0.0,True,NaN,NaN,Sunny,Christmas Market,4,False


### **Exploratory Data Analysis (EDA)**

**Ideas to explore:**
- Return rate by product category (are Toys returned more than Food?)
- Return rate by customer satisfaction score (do low satisfaction scores correlate with returns?)
- Effect of gift wrapping on returns
- Online vs in-store purchase return patterns
- Return rate during different events (Black Friday, Christmas Market)
- Weather conditions and return behavior
- Price ranges that get returned more often
- Delivery time impact on returns
- Age/gender patterns in returns
- Promotion vs non-promotion returns

**Visualization ideas for Prize 2:**
- Heatmaps of correlations
- Bar charts comparing return rates across categories
- Time series plots (returns by date/month)
- Distribution plots for numerical features
- Stacked bar charts for multi-variable comparisons

In [13]:
# Your EDA code here
# Example: train_df.groupby('Category')['ReturnFlag'].mean().sort_values()

# Check class balance
print(train_df['ReturnFlag'].value_counts())
   
# Return rate by category
train_df.groupby('Category')['ReturnFlag'].mean().sort_values()

ReturnFlag
True     4037
False    3963
Name: count, dtype: int64


Category
Clothing       0.491184
Food           0.502875
Decorations    0.503149
Toys           0.507798
Electronics    0.517512
Name: ReturnFlag, dtype: float64

### **Preprocessing Function**

**Quick wins to try:**
- Create discount percentage: `DiscountAmount / TotalPrice`
- Extract day of week categories (weekend vs weekday)
- Bin Age into groups (18-25, 26-35, 36-50, 50+)
- Create "days to Christmas" feature from Date
- Better missing value handling (use median/mode instead of -1)

**Advanced ideas:**
- Target encoding for high-cardinality features (Location, ProductName)
- Feature scaling (StandardScaler for numerical features) 
- More advanced Feature Engineering Techniques
- Interaction features (e.g., OnlineOrderFlag * DeliveryTime)

In [14]:
def preprocess_data(df, is_train=True):
    """
    Preprocess the dataset with feature engineering.
    
    Args:
        df: Input dataframe
        is_train: If True, returns (X, y). If False, returns X only.
    
    Returns:
        For training: X (features), y (target)
        For test: X (features only)
    """
    df = df.copy()
    
    # Date features
    df['Date'] = pd.to_datetime(df['Date'], format='%m/%d/%Y')
    df['Year'] = df['Date'].dt.year
    df['Month'] = df['Date'].dt.month
    df['Day'] = df['Date'].dt.day
    df['DayOfWeek'] = df['Date'].dt.dayofweek
    
    # Time features
    df['Time'] = pd.to_datetime(df['Time'], format='%H:%M:%S')
    df['Hour'] = df['Time'].dt.hour
    
    # Add your new features here!
    
    # Drop columns we don't need
    drop_cols = ['TransactionID', 'Date', 'Time', 'CustomerID', 'ProductID']
    if is_train:
        drop_cols.append('ReturnFlag')
    
    if is_train:
        y = df['ReturnFlag'].astype(int)
    
    X = df.drop(columns=[col for col in drop_cols if col in df.columns])
    
    # Fill missing values
    X = X.fillna(-1)
    
    # One-hot encode categorical variables
    X = pd.get_dummies(X, drop_first=True)
    
    if is_train:
        return X, y
    else:
        return X

### **Prepare Training Data**

### **Important Note on Validation**

Since you cannot see the true test labels, use your validation accuracy to guide your improvements. The train/validation split above mimics what will happen on the test set. A model that performs well on validation is more likely to perform well on the hidden test set.

**Strategy:**
1. Try a change (new feature, different model, etc.)
2. Check if validation accuracy improves
3. If yes, keep it. If no, try something else.
4. Once satisfied, retrain on ALL training data (as shown in the final cell) before generating your submission.

In [15]:
X, y = preprocess_data(train_df, is_train=True)

print("Encoded feature shape:", X.shape)

# Train / validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train size:", X_train.shape, "Val size:", X_val.shape)

Encoded feature shape: (8000, 55)
Train size: (6400, 55) Val size: (1600, 55)


### **Model Training**

**Models to try:**
- RandomForestClassifier (handles non-linearity well)
- XGBClassifier (powerful, often wins competitions)
- GradientBoostingClassifier
- Try ensemble methods (combine multiple models)

**Hyperparameter tuning:**
- Use GridSearchCV or RandomizedSearchCV
- Key LogisticRegression params: C, penalty
- Key RandomForest params: n_estimators, max_depth
- Key XGBoost params: learning_rate, max_depth, n_estimators

**Other tricks:**
- Use `class_weight='balanced'` if classes are imbalanced
- Try cross-validation with `cross_val_score` for more robust evaluation

In [16]:
log_reg = LogisticRegression(max_iter=1000, n_jobs=-1)
log_reg.fit(X_train, y_train)

y_val_pred = log_reg.predict(X_val)
val_acc = accuracy_score(y_val, y_val_pred)

print(f"Validation Accuracy (LogReg baseline): {val_acc:.4f}")
print("\nClassification report:\n", classification_report(y_val, y_val_pred))

Validation Accuracy (LogReg baseline): 0.4956

Classification report:
               precision    recall  f1-score   support

           0       0.49      0.44      0.46       793
           1       0.50      0.55      0.52       807

    accuracy                           0.50      1600
   macro avg       0.50      0.50      0.49      1600
weighted avg       0.50      0.50      0.49      1600



### **Generate Test Predictions**

1. Train on all training data
2. Preprocess test data
3. Align columns
4. Predict and save submission

In [17]:
# Train on ALL training data
log_reg_full = LogisticRegression(max_iter=1000, n_jobs=-1)
log_reg_full.fit(X, y)
print("Final model trained on all training data")

# Preprocess test data
X_test = preprocess_data(test_df, is_train=False)
print(f"Test data preprocessed: {X_test.shape}")

# Align columns
for col in X.columns:
    if col not in X_test.columns:
        X_test[col] = 0

X_test = X_test[X.columns]
print(f"Columns aligned: {X_test.shape}")

# Predict
test_predictions = log_reg_full.predict(X_test)

# Create submission file
submission = pd.DataFrame({
    'TransactionID': test_df['TransactionID'],
    'ReturnFlag': test_predictions
})

submission.to_csv("submission_baseline.csv", index=False)
print("Saved submission_baseline.csv\n")

submission.head(10)

Final model trained on all training data
Test data preprocessed: (2000, 55)
Columns aligned: (2000, 55)
Saved submission_baseline.csv



,TransactionID,ReturnFlag
0,8001,1
1,8002,0
2,8003,1
3,8004,1
4,8005,1
5,8006,0
6,8007,1
7,8008,0
8,8009,0
9,8010,1
